# Autobot S6E9 Nuclear 10x Final Attack

**Target**: Beat 0.94705 (Chris Deotte / real ceiling). Gap: +0.00049.

**Strategy**: 10 experiments with diverse seeds, depths, binning, and cross-model rank blends.
All use same 112-feature matrix to avoid asymmetry.

**Output**: 10 submission CSVs saved to /kaggle/working/

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

t_start = time.time()
COMP = Path('/kaggle/input/playground-series-s6e9')
OUT  = Path('/kaggle/working')

train = pd.read_csv(COMP / 'train.csv')
test  = pd.read_csv(COMP / 'test.csv')
sub   = pd.read_csv(COMP / 'sample_submission.csv')

TARGET = 'Will_Buy_EV'
y = (train[TARGET].map({'No': 0, 'Yes': 1})
     if not pd.api.types.is_numeric_dtype(train[TARGET])
     else train[TARGET]).to_numpy(np.uint8)

print(f'Train: {len(train):,}  Test: {len(test):,}  Positive rate: {y.mean():.4f}')
print(f'Columns: {list(train.columns)}')

In [ ]:
# ── Shared Feature Engineering (112-feature Jazivxt SOTA matrix) ────────────
CATEGORICAL_FEATURES = [
    'Gender', 'City_Type', 'Current_Car_Type',
    'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level'
]

def build_row_local(frame):
    out = frame.copy()
    income       = out['Annual_Income_USD'].to_numpy(dtype=np.int64)
    commute_x10  = np.round(out['Daily_Commute_km'].to_numpy(float) * 10).astype(np.int64)

    out['inc_d1']    = (income % 10).astype('int8')
    out['inc_d2']    = (income // 10 % 10).astype('int8')
    out['inc_d3']    = (income // 100 % 10).astype('int8')
    out['inc_mod100']  = (income % 100).astype('int16')
    out['inc_mod1000'] = (income % 1000).astype('int16')
    out['km_d1']     = (commute_x10 % 10).astype('int8')
    out['km_mod100'] = (commute_x10 % 100).astype('int8')

    for d in (50, 100, 250, 500, 1000, 2500, 5000):
        out[f'inc_q{d}'] = (income // d).astype('int32')
    for d in (5, 10, 25, 50):
        out[f'km_q{d}']  = (commute_x10 // d).astype('int32')

    out['is_30k_spike']        = (income == 30000).astype('int8')
    out['is_millionaire_cliff'] = (income >= 170537).astype('int8')
    out['is_dead_zone']        = ((income >= 38000) & (income <= 42000)).astype('int8')
    out['is_env_hater']        = (out['Environmental_Concern_Level'] == 1).astype('int8')

    keys = pd.DataFrame(index=out.index)
    keys['k_inc_exact'] = income.astype(str)
    keys['k_inc100']    = (income // 100).astype(str)
    keys['k_inc1000']   = (income // 1000).astype(str)
    keys['k_km_int']    = (commute_x10 // 10).astype(str)
    for col in CATEGORICAL_FEATURES + [
        'Age', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
        'Charging_Stations_Near_Work', 'Environmental_Concern_Level'
    ]:
        keys[f'k_{col}'] = out[col].astype(str).to_numpy()
    return out.reset_index(drop=True), keys.reset_index(drop=True)

def map_zero(values, mapping):
    return values.map(mapping).fillna(0.0).astype(np.float32).to_numpy()

def bin_statistics(codes, y_fit, n_bins, prior, smooth):
    sums   = np.bincount(codes, weights=y_fit, minlength=n_bins).astype(np.float64)
    counts = np.bincount(codes, minlength=n_bins).astype(np.float64)
    central = (sums + smooth * prior) / (counts + smooth)
    ls, lc  = np.r_[0.0, sums[:-1]], np.r_[0.0, counts[:-1]]
    rs, rc  = np.r_[sums[1:], 0.0], np.r_[counts[1:], 0.0]
    left  = (ls + smooth * prior) / (lc + smooth)
    right = (rs + smooth * prior) / (rc + smooth)
    kernel   = np.exp(-0.5 * (np.arange(-1, 2) / 0.8) ** 2)
    ns = np.convolve(sums, kernel, mode='same')
    nc = np.convolve(counts, kernel, mode='same')
    symmetric = (ns + smooth * kernel.sum() * prior) / (nc + smooth * kernel.sum())
    slope     = right - left
    curvature = central - 0.5 * (left + right)
    return np.column_stack([central, symmetric, left, right, slope, curvature, np.log1p(counts)]).astype(np.float32)

def income_asymmetric_full(train_inc, y, test_inc, q, smooth=10.0):
    train_inc = np.asarray(train_inc, np.float64)
    test_inc  = np.asarray(test_inc, np.float64)
    y         = np.asarray(y, np.uint8)
    prior     = float(y.mean())
    edges     = np.linspace(train_inc.min(), train_inc.max(), q + 1)
    train_codes = np.searchsorted(edges[1:-1], train_inc)
    test_codes  = np.searchsorted(edges[1:-1], test_inc)
    n_bins      = len(edges)
    pos_scale   = max(n_bins - 2, 1)
    train_feats = np.zeros((len(train_inc), 8), dtype=np.float32)
    inner = StratifiedKFold(5, shuffle=True, random_state=17)
    for fit_idx, val_idx in inner.split(train_codes, y):
        stats = bin_statistics(train_codes[fit_idx], y[fit_idx], n_bins, float(y[fit_idx].mean()), smooth)
        train_feats[val_idx, 0] = train_codes[val_idx] / pos_scale
        train_feats[val_idx, 1:] = stats[train_codes[val_idx]]
    stats = bin_statistics(train_codes, y, n_bins, prior, smooth)
    test_feats = np.column_stack([test_codes / pos_scale, stats[test_codes]]).astype(np.float32)
    return train_feats, test_feats

def build_full_features(X_tr, tr_keys, X_te, te_keys, y_arr, extra_features=False):
    X_tr = X_tr.reset_index(drop=True).copy()
    X_te = X_te.reset_index(drop=True).copy()
    tr_keys = tr_keys.reset_index(drop=True).copy()
    te_keys = te_keys.reset_index(drop=True).copy()

    inc_tr = pd.Series(X_tr['Annual_Income_USD'].to_numpy(np.int64))
    inc_te = pd.Series(X_te['Annual_Income_USD'].to_numpy(np.int64))
    inc_cnt = inc_tr.value_counts(dropna=False)
    X_tr['fq_inc'] = map_zero(inc_tr, inc_cnt)
    X_te['fq_inc'] = map_zero(inc_te, inc_cnt)

    km_tr = pd.Series(np.round(X_tr['Daily_Commute_km'].to_numpy(float) * 10).astype(np.int64))
    km_te = pd.Series(np.round(X_te['Daily_Commute_km'].to_numpy(float) * 10).astype(np.int64))
    km_cnt = km_tr.value_counts(dropna=False)
    X_tr['fq_km'] = map_zero(km_tr, km_cnt)
    X_te['fq_km'] = map_zero(km_te, km_cnt)

    for key in tr_keys.columns:
        freq = tr_keys[key].value_counts(normalize=True, dropna=False)
        X_tr[f'{key}_fe'] = map_zero(tr_keys[key], freq)
        X_te[f'{key}_fe'] = map_zero(te_keys[key], freq)

    for col in CATEGORICAL_FEATURES:
        cats = sorted(X_tr[col].astype(str).unique().tolist())
        dt   = pd.CategoricalDtype(categories=cats, ordered=False)
        X_tr[col] = X_tr[col].astype(str).astype(dt)
        X_te[col] = X_te[col].astype(str).astype(dt)

    for smooth, tag in (('auto', 'auto'), (10.0, '10'), (100.0, '100')):
        enc = TargetEncoder(shuffle=True, cv=5, smooth=smooth, random_state=42)
        enc_tr = enc.fit_transform(tr_keys, y_arr)
        enc_te = enc.transform(te_keys)
        for i, key in enumerate(tr_keys.columns):
            X_tr[f'{key}_te{tag}'] = enc_tr[:, i].astype(np.float32)
            X_te[f'{key}_te{tag}'] = enc_te[:, i].astype(np.float32)

    # Income-level residual prior (key Chris Deotte / Megayak trick)
    inc_mean = inc_tr.map(pd.Series(y_arr, dtype=float).groupby(inc_tr).mean())
    X_tr['inc_exact_prior'] = inc_mean.fillna(y_arr.mean()).astype(np.float32).values
    inc_mean_te = inc_te.map(pd.Series(y_arr, dtype=float).groupby(inc_tr).mean())
    X_te['inc_exact_prior'] = inc_mean_te.fillna(y_arr.mean()).astype(np.float32).values

    if extra_features:
        # Pairwise interaction keys
        for ca, cb in [('k_inc_exact', 'k_Subsidy_Available'),
                        ('k_inc_exact', 'k_Environmental_Concern_Level'),
                        ('k_inc_exact', 'k_Range_Anxiety_Level'),
                        ('k_inc1000',   'k_City_Type'),
                        ('k_inc1000',   'k_Current_Car_Type')]:
            combo_tr = tr_keys[ca] + '_' + tr_keys[cb]
            combo_te = te_keys[ca] + '_' + te_keys[cb]
            freq_c   = combo_tr.value_counts(normalize=True, dropna=False)
            enc_c    = TargetEncoder(shuffle=True, cv=5, smooth=10.0, random_state=42)
            enc_c.fit(combo_tr.values.reshape(-1, 1), y_arr)
            X_tr[f'interact_{ca}_{cb}_fe'] = map_zero(combo_tr, freq_c)
            X_te[f'interact_{ca}_{cb}_fe'] = map_zero(combo_te, freq_c)
            X_tr[f'interact_{ca}_{cb}_te'] = enc_c.transform(combo_tr.values.reshape(-1, 1))[:, 0].astype(np.float32)
            X_te[f'interact_{ca}_{cb}_te'] = enc_c.transform(combo_te.values.reshape(-1, 1))[:, 0].astype(np.float32)

    for res in (8192, 16384):
        tr_blk, te_blk = income_asymmetric_full(
            X_tr['Annual_Income_USD'].values, y_arr, X_te['Annual_Income_USD'].values, res)
        for j in range(1 if res == 16384 else 0, tr_blk.shape[1]):
            X_tr[f'own_income_{res}_{j}'] = tr_blk[:, j]
            X_te[f'own_income_{res}_{j}'] = te_blk[:, j]

    return X_tr, X_te

# Build base feature matrix
print('Building base feature matrix...')
X_train_raw, tr_keys = build_row_local(train.drop(columns=['id', TARGET]))
X_test_raw,  te_keys = build_row_local(test.drop(columns='id'))
X_tr_base, X_te_base = build_full_features(X_train_raw, tr_keys, X_test_raw, te_keys, y)
print(f'Base features: {X_tr_base.shape[1]}')

# Build extra-interaction feature matrix
print('Building extra-interaction feature matrix...')
X_train_raw2, tr_keys2 = build_row_local(train.drop(columns=['id', TARGET]))
X_test_raw2,  te_keys2 = build_row_local(test.drop(columns='id'))
X_tr_ext, X_te_ext = build_full_features(X_train_raw2, tr_keys2, X_test_raw2, te_keys2, y, extra_features=True)
print(f'Extended features: {X_tr_ext.shape[1]}')

In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────
def rank_pct(arr):
    """Convert probability array to percentile ranks [0,1]."""
    return pd.Series(arr).rank(pct=True).values

def save_sub(preds, name, test_ids):
    path = OUT / f'{name}.csv'
    pd.DataFrame({'id': test_ids, 'Will_Buy_EV': preds}).to_csv(path, index=False)
    n_unique = len(np.unique(preds))
    print(f'  Saved {path.name}  ({n_unique} unique values)')
    return path

ALL_RESULTS = {}  # name -> (oof_preds, test_preds, auc)
TEST_IDS = test['id'].values

LGBM_BASE_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.025,
    max_depth=3,
    num_leaves=8,
    min_child_samples=15,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.35,
    reg_alpha=0.08,
    reg_lambda=2.5,
    max_bin=255,
    feature_pre_filter=False,
    n_jobs=-1,
    verbose=-1,
    metric='auc'
)

def run_lgbm_multi_seed(name, X_tr, X_te, seeds, params=None, n_folds=5):
    p = {**LGBM_BASE_PARAMS, **(params or {})}
    print(f'\n[{name}] LGBM: {len(seeds)} seeds x {n_folds} folds = {len(seeds)*n_folds} models')
    print(f'  params: depth={p["max_depth"]} leaves={p["num_leaves"]} bin={p["max_bin"]} lr={p["learning_rate"]}')
    t0 = time.time()

    all_oof_ranks   = np.zeros(len(y), dtype=np.float64)
    all_test_ranks  = np.zeros(len(X_te), dtype=np.float64)
    all_oof_probs   = np.zeros(len(y), dtype=np.float64)

    for seed in seeds:
        seed_oof = np.zeros(len(y), dtype=np.float64)
        seed_test = np.zeros(len(X_te), dtype=np.float64)
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr, y)):
            model = LGBMClassifier(**{**p, 'random_state': seed * 100 + fold})
            model.fit(X_tr.iloc[tr_idx], y[tr_idx],
                      eval_set=[(X_tr.iloc[val_idx], y[val_idx])],
                      callbacks=[log_evaluation(-1)])
            seed_oof[val_idx] = model.predict_proba(X_tr.iloc[val_idx])[:, 1]
            seed_test += model.predict_proba(X_te)[:, 1] / n_folds
        fold_auc = roc_auc_score(y, seed_oof)
        print(f'  seed={seed} OOF AUC={fold_auc:.6f}')
        all_oof_ranks  += rank_pct(seed_oof)
        all_test_ranks += rank_pct(seed_test)
        all_oof_probs  += seed_oof

    avg_oof_ranks  = all_oof_ranks  / len(seeds)
    avg_test_ranks = all_test_ranks / len(seeds)
    avg_oof_probs  = all_oof_probs  / len(seeds)

    final_auc = roc_auc_score(y, avg_oof_ranks)
    print(f'  ==> {name} final OOF AUC (rank-avg): {final_auc:.6f}  elapsed={time.time()-t0:.0f}s')

    path = save_sub(rank_pct(avg_test_ranks), name, TEST_IDS)
    ALL_RESULTS[name] = (avg_oof_ranks, avg_test_ranks, final_auc)
    return avg_oof_ranks, avg_test_ranks

In [ ]:
# ── EXP A: 10-Seed LGBM (proven multi-seed, 2x our Exp10A) ──────────────────
oof_A, te_A = run_lgbm_multi_seed(
    'expA_lgbm_10seed',
    X_tr_base, X_te_base,
    seeds=list(range(10)),
)

In [ ]:
# ── EXP B: 20-Seed LGBM (maximum variance reduction) ─────────────────────────
oof_B, te_B = run_lgbm_multi_seed(
    'expB_lgbm_20seed',
    X_tr_base, X_te_base,
    seeds=list(range(20)),
)

In [ ]:
# ── EXP C: Deeper Trees (depth=4, leaves=16) - more capacity ─────────────────
oof_C, te_C = run_lgbm_multi_seed(
    'expC_lgbm_depth4',
    X_tr_base, X_te_base,
    seeds=list(range(5)),
    params={'max_depth': 4, 'num_leaves': 16, 'learning_rate': 0.02},
)

In [ ]:
# ── EXP D: High-Bin LGBM (max_bin=4096, 5 seeds) ────────────────────────────
oof_D, te_D = run_lgbm_multi_seed(
    'expD_lgbm_highbin4096',
    X_tr_base, X_te_base,
    seeds=list(range(5)),
    params={'max_bin': 4096, 'learning_rate': 0.02},
)

In [ ]:
# ── EXP E: Extra interaction features + 5 seeds ──────────────────────────────
oof_E, te_E = run_lgbm_multi_seed(
    'expE_lgbm_interact',
    X_tr_ext, X_te_ext,
    seeds=list(range(5)),
)

In [ ]:
# ── EXP F: XGBoost GPU (5 seeds) ─────────────────────────────────────────────
import torch
has_gpu = torch.cuda.is_available()
device  = 'cuda' if has_gpu else 'cpu'
print(f'XGBoost device: {device}')

def run_xgb_multi_seed(name, X_tr, X_te, seeds, n_folds=5):
    print(f'\n[{name}] XGBoost: {len(seeds)} seeds x {n_folds} folds')
    t0 = time.time()
    # Convert categoricals to numeric
    X_tr_n = X_tr.copy()
    X_te_n = X_te.copy()
    for col in X_tr_n.select_dtypes('category').columns:
        X_tr_n[col] = X_tr_n[col].cat.codes
        X_te_n[col] = X_te_n[col].cat.codes
    X_tr_n = X_tr_n.astype(np.float32)
    X_te_n = X_te_n.astype(np.float32)

    all_oof_ranks  = np.zeros(len(y), dtype=np.float64)
    all_test_ranks = np.zeros(len(X_te), dtype=np.float64)
    for seed in seeds:
        oof = np.zeros(len(y), dtype=np.float64)
        tst = np.zeros(len(X_te), dtype=np.float64)
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr_n, y)):
            model = XGBClassifier(
                n_estimators=3000, learning_rate=0.02,
                max_depth=6, subsample=0.8, colsample_bytree=0.4,
                tree_method='hist', device=device,
                eval_metric='auc', random_state=seed * 100 + fold,
                verbosity=0
            )
            model.fit(X_tr_n.iloc[tr_idx], y[tr_idx],
                      eval_set=[(X_tr_n.iloc[val_idx], y[val_idx])],
                      verbose=False)
            oof[val_idx] = model.predict_proba(X_tr_n.iloc[val_idx])[:, 1]
            tst += model.predict_proba(X_te_n)[:, 1] / n_folds
        fold_auc = roc_auc_score(y, oof)
        print(f'  seed={seed} OOF AUC={fold_auc:.6f}')
        all_oof_ranks  += rank_pct(oof)
        all_test_ranks += rank_pct(tst)

    avg_oof   = all_oof_ranks  / len(seeds)
    avg_test  = all_test_ranks / len(seeds)
    final_auc = roc_auc_score(y, avg_oof)
    print(f'  ==> {name} OOF AUC: {final_auc:.6f}  elapsed={time.time()-t0:.0f}s')
    save_sub(rank_pct(avg_test), name, TEST_IDS)
    ALL_RESULTS[name] = (avg_oof, avg_test, final_auc)
    return avg_oof, avg_test

oof_F, te_F = run_xgb_multi_seed('expF_xgb_5seed', X_tr_base, X_te_base, seeds=list(range(5)))

In [ ]:
# ── EXP G: CatBoost GPU ──────────────────────────────────────────────────────
def run_catboost(name, X_tr, X_te, n_folds=5):
    print(f'\n[{name}] CatBoost GPU: {n_folds} folds')
    t0 = time.time()
    X_tr_c = X_tr.copy()
    X_te_c = X_te.copy()
    cat_idx = [X_tr_c.columns.get_loc(c)
               for c in X_tr_c.select_dtypes('category').columns]
    for col in X_tr_c.select_dtypes('category').columns:
        X_tr_c[col] = X_tr_c[col].cat.codes.astype(str)
        X_te_c[col] = X_te_c[col].cat.codes.astype(str)

    oof   = np.zeros(len(y), dtype=np.float64)
    tst   = np.zeros(len(X_te), dtype=np.float64)
    skf   = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr_c, y)):
        model = CatBoostClassifier(
            iterations=5000, learning_rate=0.03,
            depth=6, eval_metric='AUC',
            task_type='GPU' if has_gpu else 'CPU',
            random_seed=42 + fold, verbose=0,
            cat_features=cat_idx
        )
        model.fit(X_tr_c.iloc[tr_idx], y[tr_idx],
                  eval_set=(X_tr_c.iloc[val_idx], y[val_idx]))
        oof[val_idx] = model.predict_proba(X_tr_c.iloc[val_idx])[:, 1]
        tst += model.predict_proba(X_te_c)[:, 1] / n_folds
        print(f'  fold {fold+1}/{n_folds} AUC={roc_auc_score(y[val_idx], oof[val_idx]):.6f}')

    final_auc = roc_auc_score(y, oof)
    print(f'  ==> {name} OOF AUC: {final_auc:.6f}  elapsed={time.time()-t0:.0f}s')
    save_sub(rank_pct(tst), name, TEST_IDS)
    ALL_RESULTS[name] = (rank_pct(oof), rank_pct(tst), final_auc)
    return rank_pct(oof), rank_pct(tst)

oof_G, te_G = run_catboost('expG_catboost_5k', X_tr_base, X_te_base)

In [ ]:
# ── BLENDS (Exps H, I, J) ─────────────────────────────────────────────────────

def rank_blend(oof_list, te_list, weights, name):
    """Weighted rank average blend."""
    weights = np.array(weights, dtype=np.float64)
    weights /= weights.sum()
    oof_blended = sum(w * rank_pct(o) for w, o in zip(weights, oof_list))
    te_blended  = sum(w * rank_pct(t) for w, t in zip(weights, te_list))
    final_auc = roc_auc_score(y, oof_blended)
    print(f'\n[{name}] Rank blend OOF AUC: {final_auc:.6f}')
    save_sub(rank_pct(te_blended), name, TEST_IDS)
    ALL_RESULTS[name] = (oof_blended, te_blended, final_auc)
    return oof_blended, te_blended

# EXP H: Best LGBM (20-seed) + XGB blend
oof_H, te_H = rank_blend(
    [oof_B, oof_F], [te_B, te_F],
    [0.6, 0.4],
    'expH_lgbm20_xgb_blend'
)

# EXP I: LGBM(20) + XGB + CatBoost three-way
oof_I, te_I = rank_blend(
    [oof_B, oof_F, oof_G], [te_B, te_F, te_G],
    [0.5, 0.3, 0.2],
    'expI_lgbm20_xgb_cat_3way'
)

# EXP J: Grand ensemble: 20-seed + high-bin + interact + XGB + CAT
oof_J, te_J = rank_blend(
    [oof_B, oof_D, oof_E, oof_F, oof_G],
    [te_B,  te_D,  te_E,  te_F,  te_G],
    [0.35, 0.20, 0.15, 0.20, 0.10],
    'expJ_grand_ensemble'
)

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('\n' + '='*70)
print('FINAL SUMMARY — ALL 10 EXPERIMENTS')
print('='*70)
print(f'{"Experiment":<35} {"OOF AUC":>10} {"File"}')
print('-'*70)
ranked = sorted(ALL_RESULTS.items(), key=lambda kv: kv[1][2], reverse=True)
for name, (_, _, auc) in ranked:
    print(f'{name:<35} {auc:>10.6f}  /kaggle/working/{name}.csv')

print(f'\nTotal elapsed: {(time.time() - t_start)/60:.1f} minutes')
print('\nAll 10 CSVs are ready for submission.')
print('Download and submit the top-ranked ones first!')
print('\nAvailable files:')
for f in sorted(OUT.glob('exp*.csv')):
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')